<a href="https://colab.research.google.com/github/nibaskumar93n-debug/Morphoinformatics/blob/main/FastQ_to_count_matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/current/sratoolkit.current-ubuntu64.tar.gz
!tar -xzvf sratoolkit.current-ubuntu64.tar.gz

--2026-05-09 06:44:53--  https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/current/sratoolkit.current-ubuntu64.tar.gz
Resolving ftp-trace.ncbi.nlm.nih.gov (ftp-trace.ncbi.nlm.nih.gov)... 130.14.250.10, 130.14.250.11, 2607:f220:41e:250::7, ...
Connecting to ftp-trace.ncbi.nlm.nih.gov (ftp-trace.ncbi.nlm.nih.gov)|130.14.250.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 89143120 (85M) [application/x-gzip]
Saving to: ‘sratoolkit.current-ubuntu64.tar.gz’

sratoolkit.current- 100%[===================>]  85.01M   216MB/s    in 0.4s    

2026-05-09 06:44:54 (216 MB/s) - ‘sratoolkit.current-ubuntu64.tar.gz’ saved [89143120/89143120]

sratoolkit.3.4.1-ubuntu64/
sratoolkit.3.4.1-ubuntu64/bin/
sratoolkit.3.4.1-ubuntu64/bin/abi-dump
sratoolkit.3.4.1-ubuntu64/bin/abi-dump.3
sratoolkit.3.4.1-ubuntu64/bin/abi-load
sratoolkit.3.4.1-ubuntu64/bin/abi-load.3
sratoolkit.3.4.1-ubuntu64/bin/align-info
sratoolkit.3.4.1-ubuntu64/bin/align-info.3
sratoolkit.3.4.1-ubuntu64/bin/bam-load
sr

In [2]:
!ls /content/

sample_data  sratoolkit.3.4.1-ubuntu64	sratoolkit.current-ubuntu64.tar.gz


In [3]:
!ls /content/ | grep sratoolkit

sratoolkit.3.4.1-ubuntu64
sratoolkit.current-ubuntu64.tar.gz


In [4]:
import os
os.environ['PATH'] = '/content/sratoolkit.3.4.1-ubuntu64/bin:' + os.environ['PATH']
print("PATH updated!")

PATH updated!


In [5]:
!which fastq-dump
!which fasterq-dump
!fastq-dump --version
!fasterq-dump --version

/content/sratoolkit.3.4.1-ubuntu64/bin/fastq-dump
/content/sratoolkit.3.4.1-ubuntu64/bin/fasterq-dump
fastq-dump : 3.4.1

fasterq-dump : 3.4.1



In [6]:
!mkdir -p /content/sra_data
!mkdir -p /content/fastq_files

In [7]:
# Replace with your actual SRR numbers
srr_list = [
    "SRR12185519",
]

In [8]:
for srr in srr_list:
    print(f"Prefetching {srr}...")
    !prefetch {srr} --output-directory /content/sra_data
    print(f"{srr} done!")

Prefetching SRR12185519...
2026-05-09T06:45:47 prefetch.3.4.1: 1) Resolving 'SRR12185519'...
2026-05-09T06:45:48 prefetch.3.4.1: Current preference is set to retrieve SRA Normalized Format files with full base quality scores
2026-05-09T06:45:48 prefetch.3.4.1: 1) Downloading 'SRR12185519'...
2026-05-09T06:45:48 prefetch.3.4.1:  SRA Normalized Format file is being retrieved
2026-05-09T06:45:48 prefetch.3.4.1:  Downloading via HTTPS...
2026-05-09T06:48:25 prefetch.3.4.1:  HTTPS download succeed
2026-05-09T06:48:57 prefetch.3.4.1:  'SRR12185519' is valid: 7332633080 bytes were streamed from 7332623477
2026-05-09T06:48:57 prefetch.3.4.1: 1) 'SRR12185519' was downloaded successfully
2026-05-09T06:48:57 prefetch.3.4.1: 1) Resolving 'SRR12185519's dependencies...
2026-05-09T06:49:02 prefetch.3.4.1: 'SRR12185519' has 101 unresolved dependencies
2026-05-09T06:49:02 prefetch.3.4.1: 2) Resolving 'ncbi-acc:GL000008.2?vdb-ctx=refseq'...
2026-05-09T06:49:02 prefetch.3.4.1: 2) Downloading 'ncbi-acc:G

In [15]:
for srr in srr_list:
    print(f"Converting {srr}...")
    !fastq-dump /content/sra_data/{srr}/{srr}.sra \
        --outdir /content/fastq_files \
        --split-3 \
        --gzip
    print(f"{srr} done!")

Converting SRR12185519...
Read 165788892 spots for /content/sra_data/SRR12185519/SRR12185519.sra
Written 165788892 spots for /content/sra_data/SRR12185519/SRR12185519.sra
SRR12185519 done!


In [17]:
!ls -lh /content/fastq_files/

total 7.2G
-rw-r--r-- 1 root root 7.2G May  9 08:58 SRR12185519.fastq.gz


In [18]:
import os

# File is single-end so no _1 or _2 suffix
os.rename(
    '/content/fastq_files/SRR12185519.fastq.gz',
    '/content/fastq_files/sample1_S1_L001_R1_001.fastq.gz'
)
print("Renamed successfully!")
!ls -lh /content/fastq_files/

Renamed successfully!
total 7.2G
-rw-r--r-- 1 root root 7.2G May  9 08:58 sample1_S1_L001_R1_001.fastq.gz


In [19]:
!ls -lh /content/fastq_files/

total 7.2G
-rw-r--r-- 1 root root 7.2G May  9 08:58 sample1_S1_L001_R1_001.fastq.gz


In [20]:
import os
import subprocess

files = sorted(os.listdir('/content/fastq_files/'))
for f in files:
    result = subprocess.run(
        f"zcat /content/fastq_files/{f} | wc -l",
        shell=True, capture_output=True, text=True
    )
    reads = int(result.stdout.strip()) // 4
    print(f"{f}: {reads:,} reads")

sample1_S1_L001_R1_001.fastq.gz: 165,788,892 reads


In [21]:
!cellranger count \
    --id=sample1_output \
    --transcriptome=/content/refdata-gex-GRCh38-2020-A \
    --fastqs=/content/fastq_files \
    --sample=sample1 \
    --localcores=4 \
    --localmem=13

/bin/bash: line 1: cellranger: command not found
